In [17]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

In [18]:
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                text+=page_text
        if text.strip():
            return text.strip()

    except Exception as e:
        print(f"direct text extraction as failed : {e}")
    
    #if pdf is image based

    print("Falling back to OCR for image-based pdf")
    try:
        images = convert_from_path(pdf_path)
        for image in images:
            image_text = pytesseract.image_to_string(image)
            text+=image_text + "\n"
    except Exception as e:
        print(f'OCR Failed : {e}')
    
    return text.strip()

In [ ]:
pdf_path = "Fathimasherin_cv.pdf"
resume_text = extract_text_from_pdf(pdf_path)
print("Extracted text from resume")
print(resume_text)

In [ ]:
import google.generativeai as genai
import os
from dotenv import load_dotenv  
load_dotenv()
genai.configure(api_key=os.getenv("api_key"))
model = genai.GenerativeModel("gemini-3-flash-preview")

In [21]:
def analyse_resume(resume_text,job_description=None):
    if not resume_text:
        return("Resume text is required")
    model = genai.GenerativeModel("gemini-3-flash-preview")

    base_prompt = f'''You are an experienced HR with Technical Experience in the field of any one job role from Data Science, Data Analyst,
    DevOPS, Machine Learning Engineer, Prompt Engineer, AI Engineer, Full Stack Web Development, Big Data Engineering, Marketing Analyst,
    Human Resource Manager, Software Developer your task is to review the provided resume.
    Please share your professional evaluation on whether the candidate's profile aligns with the role. ALso mention Skills he already have
    and siggest some skills to imorve his resume, alos suggest some course he might take to improve the skills Highlight the strengths and 
    weaknesses.
    Resume :
    {resume_text}
    '''

    if job_description:
        base_prompt+= f'''
        Additionally,compare this resume to the folowing Job Description:

        Job Description:{job_description}

        Highlight the stregth and weaaknesses of the applicant in relation To the specified job requirements.
    
        '''

    respose = model.generate_content(base_prompt)
    analysis = respose.text.strip()
    return analysis

In [ ]:
print(analyse_resume(resume_text))